## DATA INSPECTION

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [2]:
# === Load all available ColorMNIST datasets ===
import torch, os, glob

# auto-detect all generated datasets under data/
DATASETS = sorted(glob.glob("data/cmnist*_clean"))
print("Available datasets:")
for i, d in enumerate(DATASETS):
    print(f"  [{i}] {d}")

# --- Select which dataset to inspect ---
SELECT = 12 # <-- change this index to pick dataset
DATA_DIR = DATASETS[SELECT]
print(f"\n📦 Using dataset: {DATA_DIR}")

# --- Load data files ---
Dll_samples = torch.load(os.path.join(DATA_DIR, "dll_samples.pkl"))
omega_labels = torch.load(os.path.join(DATA_DIR, "intervention_mapping.pkl"))

# convenience aliases (for backward compatibility)
Dll_samples[None] = Dll_samples.get("obs", next(iter(Dll_samples.values())))
omega = {k: k for k in omega_labels.keys()}

# preview structure
print(f"DLL keys: {list(Dll_samples.keys())[:5]}")


Available datasets:
  [0] data/cmnist16_c00_clean
  [1] data/cmnist16_c25_clean
  [2] data/cmnist16_c50_clean
  [3] data/cmnist16_c85_clean
  [4] data/cmnist32_c00_clean
  [5] data/cmnist32_c25_clean
  [6] data/cmnist32_c50_clean
  [7] data/cmnist32_c85_clean
  [8] data/cmnist8_c00_clean
  [9] data/cmnist8_c25_clean
  [10] data/cmnist8_c50_clean
  [11] data/cmnist8_c85_clean
  [12] data/cmnist_28_c00_clean

📦 Using dataset: data/cmnist_28_c00_clean
DLL keys: ['obs', 'D=6', 'D=8', 'D=4', 'C=7']


In [3]:
# === Interactive CMNIST inspector (widgets + live output) ===
import os, glob, torch, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- discover datasets ---
DATASETS = sorted(glob.glob("data/cmnist*_clean"))
if not DATASETS:
    raise RuntimeError("No datasets found under data/. Generate with run_cmnist_batch.sh first.")

# UI controls
ds_dropdown   = widgets.Dropdown(
    options=[(os.path.basename(d), d) for d in DATASETS],
    value=DATASETS[0], description="Dataset:", style={"description_width":"120px"}, layout=widgets.Layout(width="400px")
)
iv_dropdown   = widgets.Dropdown(
    options=[], description="Intervention:", style={"description_width":"120px"}, layout=widgets.Layout(width="400px")
)
view_toggle   = widgets.ToggleButtons(
    options=[("Triplets (images)","triplets"), ("HL hist 4-way","hist")],
    value="triplets", description="", layout=widgets.Layout(width="400px")
)
n_slider      = widgets.IntSlider(
    value=5, min=3, max=10, step=1, description="# samples:", style={"description_width":"120px"}, layout=widgets.Layout(width="400px")
)
out = widgets.Output()

# state
Dll_samples = None
Dhl_samples = None
omega_labels = None
current_ds = None
current_iv = None

def _load_dataset(folder):
    global Dll_samples, Dhl_samples, omega_labels
    Dll_samples = torch.load(os.path.join(folder, "dll_samples.pkl"))
    Dhl_samples = torch.load(os.path.join(folder, "dhl_samples.pkl"))
    omega_labels = torch.load(os.path.join(folder, "intervention_mapping.pkl"))

def _render():
    with out:
        clear_output(wait=True)
        if Dll_samples is None or omega_labels is None:
            print("No dataset loaded.")
            return

        # pick observational + chosen intervention
        obs_images, obs_shapes, obs_digits, obs_colors = Dll_samples["obs"]
        iota = iv_dropdown.value
        if iota not in Dll_samples:
            print(f"Intervention '{iota}' not found in Dll_samples keys: {list(Dll_samples.keys())}")
            return
        int_images, int_shapes, int_digits, int_colors = Dll_samples[iota]

        if view_toggle.value == "triplets":
            n = min(n_slider.value, len(obs_digits), len(int_digits))
            fig, axes = plt.subplots(3, n, figsize=(3*n, 7))
            if n == 1:
                axes = np.array([[axes[0]],[axes[1]],[axes[2]]])  # normalize shape

            for i in range(n):
                # Row 1: original grayscale ([-1,1] -> [0,1])
                shape = obs_shapes[i].squeeze()
                axes[0, i].imshow((shape + 1) / 2, cmap="gray")
                axes[0, i].set_title(f"Original\nDigit: {obs_digits[i].item()}")
                axes[0, i].axis("off")

                # Row 2: observational colored ([0,1])
                img = obs_images[i].permute(1, 2, 0).clamp(0, 1)
                axes[1, i].imshow(img)
                axes[1, i].set_title(f"Obs\nD: {obs_digits[i].item()}  C: {obs_colors[i].item()}")
                axes[1, i].axis("off")

                # Row 3: intervention colored ([0,1])
                int_img = int_images[i].permute(1, 2, 0).clamp(0, 1)
                axes[2, i].imshow(int_img)
                axes[2, i].set_title(f"Intv {iota}\nD: {int_digits[i].item()}  C: {int_colors[i].item()}")
                axes[2, i].axis("off")

            axes[0, 0].text(-0.3, 0.5, 'Original MNIST', transform=axes[0, 0].transAxes,
                            rotation=90, va='center', ha='center', fontsize=12, fontweight='bold')
            axes[1, 0].text(-0.3, 0.5, 'Observational cMNIST', transform=axes[1, 0].transAxes,
                            rotation=90, va='center', ha='center', fontsize=12, fontweight='bold')
            axes[2, 0].text(-0.3, 0.5, 'Interventional cMNIST', transform=axes[2, 0].transAxes,
                            rotation=90, va='center', ha='center', fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.show()

        else:  # "hist" — HL four-way comparison
            # Utility to get last HL column (Image_ mean intensity)
            def hl_last_col(entry): return entry[:, -1].numpy()
            def get(key): return Dhl_samples[key]

            # Pick standard quartet
            obs_averages       = hl_last_col(get("obs"))
            # Default IVs (present in your generation list); if missing, adjust here
            digit_key = "D=8" if "D=8" in Dhl_samples else list(Dhl_samples.keys())[1]
            color_key = "C=0" if "C=0" in Dhl_samples else [k for k in Dhl_samples if k.startswith("C=")][0]
            both_key  = "D=8,C=0" if "D=8,C=0" in Dhl_samples else [k for k in Dhl_samples if "," in k][0]

            int_digit_avgs     = hl_last_col(get(digit_key))
            int_color_avgs     = hl_last_col(get(color_key))
            int_both_averages  = hl_last_col(get(both_key))

            plt.figure(figsize=(12, 7))
            plt.hist(obs_averages,      bins=50, alpha=0.50, density=True, label='Observational', color='blue')
            plt.hist(int_digit_avgs,    bins=50, alpha=0.70, density=True, label=f'do({digit_key})', color='orange')
            plt.hist(int_color_avgs,    bins=50, alpha=0.70, density=True, label=f'do({color_key})', color='green')
            plt.hist(int_both_averages, bins=50, alpha=0.85, density=True, label=f'do({both_key})', color='purple')
            plt.title(f'HL "Image_" Distribution — {os.path.basename(ds_dropdown.label)}')
            plt.xlabel('Mean pixel intensity (Image_)'); plt.ylabel('Density')
            plt.grid(ls='--', alpha=0.5); plt.legend()
            plt.show()

            # Stats
            print("--- Distribution Statistics for HL 'Image_' ---")
            print("Observational:")
            print(f"  Mean: {obs_averages.mean():.4f} | Std: {obs_averages.std():.4f}")
            print(f"{digit_key}:")
            print(f"  Mean: {int_digit_avgs.mean():.4f} | Std: {int_digit_avgs.std():.4f}")
            print(f"{color_key}:")
            print(f"  Mean: {int_color_avgs.mean():.4f} | Std: {int_color_avgs.std():.4f}")
            print(f"{both_key}:")
            print(f"  Mean: {int_both_averages.mean():.4f} | Std: {int_both_averages.std():.4f}")

def _on_change_dataset(change):
    folder = change["new"]
    _load_dataset(folder)
    all_labels = list(omega_labels.keys())
    iv_dropdown.options = all_labels
    # choose a sensible default intervention (digital one if present)
    iv_dropdown.value = "D=6" if "D=6" in all_labels else all_labels[1]
    _render()

def _on_change_intervention(change):
    _render()

def _on_change_controls(change):
    _render()

# wire events
ds_dropdown.observe(_on_change_dataset, names="value")
iv_dropdown.observe(_on_change_intervention, names="value")
view_toggle.observe(_on_change_controls, names="value")
n_slider.observe(_on_change_controls, names="value")

# first load + display
_load_dataset(ds_dropdown.value)
iv_dropdown.options = list(omega_labels.keys())
iv_dropdown.value = "D=6" if "D=6" in iv_dropdown.options else iv_dropdown.options[1]
controls = widgets.VBox([ds_dropdown, iv_dropdown, widgets.HBox([view_toggle, n_slider])])
display(controls, out)
_render()


Output()